# Qwen2.5-0.5B LoRA Fine-Tune - T4 x2 1 Hour - Bangla (My-ai-1)
Trained on `My-ai-1/data/processed/training_data.jsonl` (2.19GB, 20k capped) → Qwen2.5-0.5B-Instruct FP16 LoRA r=32 1500 steps (~1.2 epochs) → Q3_K_M ~355MB fits Render Free 512MB + Neon Free.
Set Kaggle Notebook: **Accelerator T4 x2, Internet ON, Add Input → My-ai-1**

In [ ]:
!pip install -q -U transformers==4.44.2 peft==0.12.0 accelerate==0.33.0 sentencepiece huggingface_hub
# DO NOT install bitsandbytes here - FP16 LoRA avoids triton.ops mismatch
import torch
print("CUDA", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p=torch.cuda.get_device_properties(i)
        print(f"GPU {i}: {p.name} {p.total_memory/1e9:.1f}GB")
!nvidia-smi
print("If first run, do Kernel -> Restart -> re-run this cell once more, then run next cells")

In [ ]:
import glob, os, shutil, zipfile
print("=== /kaggle/input ===")
!ls -R /kaggle/input 2>&1 | head -n 100
os.makedirs("/kaggle/working/data", exist_ok=True)
zips = glob.glob("/kaggle/input/**/*.zip", recursive=True)
print(f"zips: {zips}")
for zp in zips:
    with zipfile.ZipFile(zp, "r") as z: z.extractall("/kaggle/working/data")
for src in glob.glob("/kaggle/input/**/*", recursive=True):
    if os.path.isfile(src) and not src.endswith(".zip"):
        rel = os.path.relpath(src, "/kaggle/input")
        dst = os.path.join("/kaggle/working/data", rel)
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        if not os.path.exists(dst): shutil.copy2(src, dst)
print("=== /kaggle/working/data ===")
!find /kaggle/working/data -type f | head -n 20
!du -h /kaggle/working/data 2>&1 | tail -n 20
candidates = glob.glob("/kaggle/working/data/**/training_data.jsonl", recursive=True)
print(candidates)
assert candidates, "training_data.jsonl not found"

In [ ]:
import json, hashlib, random, glob, os
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType
import torch

MAX_EXAMPLES=20000
MODEL_NAME="Qwen/Qwen2.5-0.5B-Instruct"
src = glob.glob("/kaggle/working/data/**/training_data.jsonl", recursive=True)[0]
print(src, f"{os.path.getsize(src)/1e9:.2f}GB")
items, seen=[], set()
with open(src, encoding="utf-8") as f:
    for line in f:
        if not line.strip(): continue
        obj=json.loads(line)
        if "messages" not in obj: continue
        k=hashlib.sha256(json.dumps(obj, ensure_ascii=False).encode()).hexdigest()
        if k in seen: continue
        seen.add(k)
        items.append(obj)
        if len(items)>=MAX_EXAMPLES: break
random.Random(42).shuffle(items)
print(f"Kept {len(items)}")
ds=Dataset.from_list(items)
split=ds.train_test_split(test_size=0.01, seed=42)
train_ds, eval_ds=split["train"], split["test"]
print(len(train_ds), len(eval_ds))
tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None: tokenizer.pad_token=tokenizer.eos_token
def fmt(ex):
    txt=tokenizer.apply_chat_template(ex["messages"], tokenize=False)
    t=tokenizer(txt, truncation=True, max_length=512, padding="max_length")
    t["labels"]=t["input_ids"].copy()
    return t
train_ds=train_ds.map(fmt, remove_columns=train_ds.column_names, desc="tok train")
eval_ds=eval_ds.map(fmt, remove_columns=eval_ds.column_names, desc="tok eval")
model=AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map="auto", trust_remote_code=True)
model.config.use_cache=False
lora=LoraConfig(task_type=TaskType.CAUSAL_LM, r=32, lora_alpha=64, lora_dropout=0.05, target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])
model=get_peft_model(model, lora)
model.enable_input_require_grads()
model.print_trainable_parameters()

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"]="false"
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
OUTPUT_DIR="/kaggle/working/models/qwen500M-bangla-lora"
args=TrainingArguments(
    output_dir=OUTPUT_DIR, num_train_epochs=1, max_steps=1500,
    per_device_train_batch_size=4, gradient_accumulation_steps=4,
    learning_rate=2e-4, warmup_ratio=0.03, logging_steps=10,
    save_steps=500, save_total_limit=2, eval_strategy="steps",
    eval_steps=500, fp16=True, optim="adamw_torch",
    gradient_checkpointing=False, report_to="none", dataloader_num_workers=2, group_by_length=True
)
trainer=Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=eval_ds, data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False))
trainer.train()
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved to {OUTPUT_DIR}")
!ls -lh /kaggle/working/models/qwen500M-bangla-lora
!zip -r /kaggle/working/qwen500M-bangla-lora.zip /kaggle/working/models/qwen500M-bangla-lora && ls -lh /kaggle/working/*.zip
from peft import PeftModel
print("Merging...")
base=AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, trust_remote_code=True, device_map="auto")
peft_model=PeftModel.from_pretrained(base, OUTPUT_DIR)
merged=peft_model.merge_and_unload()
merged.save_pretrained("/kaggle/working/models/merged")
tokenizer.save_pretrained("/kaggle/working/models/merged")
import torch
inputs=tokenizer("হ্যালো, তুমি কে? বাংলায় বলো।", return_tensors="pt").to(merged.device)
with torch.no_grad():
    out=merged.generate(**inputs, max_new_tokens=100, temperature=0.7, top_p=0.9, do_sample=True)
print(tokenizer.decode(out[0], skip_special_tokens=True))
print("Quantizing to Q3_K_M for Render Free (210-355MB, 24 layers)...")
!git clone --depth 1 https://github.com/ggerganov/llama.cpp.git /kaggle/working/llama.cpp -q
!pip install -q -r /kaggle/working/llama.cpp/requirements.txt
!python /kaggle/working/llama.cpp/convert_hf_to_gguf.py /kaggle/working/models/merged --outfile /kaggle/working/model-f16.gguf --outtype f16
!cmake -B /kaggle/working/llama.cpp/build -S /kaggle/working/llama.cpp -DCMAKE_BUILD_TYPE=Release -DLLAMA_CURL=OFF -q && cmake --build /kaggle/working/llama.cpp/build -j4 -q
!/kaggle/working/llama.cpp/build/bin/llama-quantize /kaggle/working/model-f16.gguf /kaggle/working/qwen2.5-0.5b-bangla-Q3_K_M.gguf Q3_K_M && ls -lh /kaggle/working/*.gguf
print("Done. Download qwen500M-bangla-lora.zip + qwen2.5-0.5b-bangla-Q3_K_M.gguf")